1 - Importação dos pacotes e bibliotecas

In [15]:
# Usada para manipulação de dados
import pandas as pd

import numpy as np

# Usado para tranformar strings em dados numéricos
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Módulos do Scikit-learn usado para algoritimos
from sklearn.ensemble import RandomForestRegressor

# Módulo do Scikit-learn usado para separar o conjunto de dados.
from sklearn.model_selection import train_test_split

# Módulos do Scikit-learn usado para avaliação estatística.
from sklearn.metrics  import mean_absolute_error
from sklearn.metrics  import mean_squared_error
from sklearn.metrics  import r2_score

# usado para salvar o modelo.
import joblib

2 - Ingenstão dos dados de entrega

In [16]:
# Define o Caminho dos arquivos.
csv_customers = "../data/olist_customers_dataset.csv"
csv_order_itens = "../data/olist_order_items_dataset.csv"
csv_orders = "../data/olist_orders_dataset.csv"
csv_products = "../data/olist_products_dataset.csv"
csv_sellers = "../data/olist_sellers_dataset.csv"
csv_geolocation = "../data/olist_geolocation_dataset.csv"

# Carrega os arquivos CSV como DataFrames
df_costumers = pd.read_csv(csv_customers)
df_order_itens = pd.read_csv(csv_order_itens)
df_orders = pd.read_csv(csv_orders)
df_products = pd.read_csv(csv_products)
df_sellers = pd.read_csv(csv_sellers)
df_geolocation = pd.read_csv(csv_geolocation)

# Junta os dataframes em um unico dataframe para utilizar no machine learning.
df = df_orders.merge(df_order_itens, on="order_id")
df = df.merge(df_costumers, on="customer_id")
df = df.merge(df_sellers, on='seller_id')
df = df.merge(df_products, on="product_id")


# Exibe o dataframe
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,seller_city,seller_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,maua,SP,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,belo horizonte,SP,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,guariba,SP,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,belo horizonte,MG,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,...,mogi das cruzes,SP,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


3 - Tratamento dos dados

In [17]:
# pega a latitude e longitude pelo CEP

# remove as colunas de cidade e estado, pois não são necessárias para o modelo.
df_geolocation =  df_geolocation.drop(columns=["geolocation_city", "geolocation_state"])

# AGRUPA ANTES
df_geolocation = df_geolocation.groupby(
    "geolocation_zip_code_prefix"
).mean().reset_index()

# cliente
df = df.merge(
    df_geolocation,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

df = df.rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
})
# vendedor
df = df.merge(
    df_geolocation,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)
df = df.rename(columns={
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})
df = df.drop(columns=["geolocation_zip_code_prefix_x", "geolocation_zip_code_prefix_y"], axis=1)
df.head() 

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_lat,customer_lng,seller_lat,seller_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,-23.576983,-46.587161,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,178.0,1.0,400.0,19.0,13.0,19.0,-12.177924,-44.660711,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,232.0,1.0,420.0,24.0,19.0,21.0,-16.745150,-48.514783,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,468.0,3.0,450.0,30.0,10.0,20.0,-5.774190,-35.271143,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,...,316.0,4.0,250.0,51.0,15.0,15.0,-23.676370,-46.514627,-23.543395,-46.262086


In [18]:
# funções usadas no treino
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371  # raio da Terra em km
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [19]:
# Cria a coluna de distância usando as funções

df["distance"] = calculate_distance(
    df["customer_lat"],
    df["customer_lng"], 
    df["seller_lat"],
    df["seller_lng"]
)


In [20]:
# Converte as datas para datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

# Cria a Variável alvo: diferença em dias entre a entrega e a compra.
df['prazo_entrega_dias'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp'])

# Converte para um dado numérico.
df['prazo_entrega_dias'] = df['prazo_entrega_dias'].dt.days

# Remove os outliers de tempo de entrega.
df = df[df['prazo_entrega_dias'] <= 30]

# Cria colunas com o dia da semana de compra, o mês e a hora.
df['order_weekday'] = df['order_purchase_timestamp'].dt.weekday # 0 = segunda.
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_hour'] = df['order_purchase_timestamp'].dt.hour

# Exclui as colunas de data de compra e data de entrega.
df = df.drop(['order_purchase_timestamp', 'order_delivered_customer_date'] , axis=1)

In [21]:
# Tira todas as colunas irrelevantes para o aprendizado de maquina.
df = df.drop(['order_id', 'customer_id', 'customer_state', 'seller_state','order_status', 'product_id', 'order_item_id', 'seller_id', 'customer_unique_id', 'product_name_lenght', 'product_description_lenght', "product_photos_qty", "order_approved_at", 'order_delivered_carrier_date', 'order_estimated_delivery_date', 'shipping_limit_date', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng'], axis=1)


In [23]:
# Remove as linhas que contenham algum valor nulo.
df = df.dropna()

# Trasforma as colunas com strings em valores numeros para o aprendizado de maquina.
colunas_categoricas = ['customer_city', 'seller_city', 'product_category_name']

encoders = {}
for col in colunas_categoricas:
#     df[col] = le.fit_transform(df[col].astype(str))
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le


In [24]:
# Exibe o dataframe
display(df)

# Exibe informações detalhadas sobre a estrutura dos dados.
display(df.info())

# Exibe a quantidade de valores nulos por coluna.
print(df.isnull().sum())


,price,freight_value,customer_zip_code_prefix,customer_city,seller_zip_code_prefix,seller_city,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,distance,prazo_entrega_dias,order_weekday,order_month,order_hour
0,29.99,8.72,3149,3420,9350,317,72,500.0,19.0,8.0,13.0,18.576110,8.0,0,10,10
1,118.70,22.76,47813,396,31570,61,62,400.0,19.0,13.0,19.0,851.495069,13.0,1,7,20
2,159.90,19.22,75265,3854,14840,211,8,420.0,24.0,19.0,21.0,514.410666,9.0,2,8,8
3,45.00,27.20,59296,3301,31842,61,63,450.0,30.0,10.0,20.0,1822.226336,13.0,5,11,19
4,19.90,8.72,9195,3211,8752,327,59,250.0,51.0,15.0,15.0,29.676625,2.0,1,2,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,174.90,20.10,11722,2848,17602,553,9,4950.0,40.0,10.0,40.0,474.120037,22.0,1,2,12
112646,205.99,65.02,45920,2413,8290,495,29,13300.0,32.0,90.0,22.0,967.847297,24.0,6,8,14
112647,179.99,40.59,28685,1840,37175,224,44,6550.0,20.0,20.0,20.0,370.404482,17.0,0,1,21
112648,179.99,40.59,28685,1840,37175,224,44,6550.0,20.0,20.0,20.0,370.404482,17.0,0,1,21


<class 'pandas.core.frame.DataFrame'>
Index: 103668 entries, 0 to 112649
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   price                     103668 non-null  float64
 1   freight_value             103668 non-null  float64
 2   customer_zip_code_prefix  103668 non-null  int64  
 3   customer_city             103668 non-null  int64  
 4   seller_zip_code_prefix    103668 non-null  int64  
 5   seller_city               103668 non-null  int64  
 6   product_category_name     103668 non-null  int64  
 7   product_weight_g          103668 non-null  float64
 8   product_length_cm         103668 non-null  float64
 9   product_height_cm         103668 non-null  float64
 10  product_width_cm          103668 non-null  float64
 11  distance                  103668 non-null  float64
 12  prazo_entrega_dias        103668 non-null  float64
 13  order_weekday             103668 non-null  int32 

None

price                       0
freight_value               0
customer_zip_code_prefix    0
customer_city               0
seller_zip_code_prefix      0
seller_city                 0
product_category_name       0
product_weight_g            0
product_length_cm           0
product_height_cm           0
product_width_cm            0
distance                    0
prazo_entrega_dias          0
order_weekday               0
order_month                 0
order_hour                  0
dtype: int64


4 - Separação dos dados     

In [25]:
# Divide os dados para treinamento e teste.
df_treino, df_teste = train_test_split(df, test_size=0.2, random_state=42)

# Define o alvo e os preditores de treinamento.
y_treino = df_treino['prazo_entrega_dias']
x_treino = df_treino[['price','customer_zip_code_prefix', 'customer_city', 'seller_zip_code_prefix', 'seller_city', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'order_weekday', 'order_month', 'order_hour', 'distance']]

# Define o alvo e os preditores de teste.
y_teste = df_teste['prazo_entrega_dias']
x_teste = df_teste[['price','customer_zip_code_prefix', 'customer_city', 'seller_zip_code_prefix', 'seller_city', 'product_category_name', 'product_weight_g','product_length_cm', 'product_height_cm', 'product_width_cm', 'order_weekday', 'order_month', 'order_hour', 'distance']]

5 - Treinamento dos modelos de machine learning

In [26]:
# Cria o modelo de Random Forest
RF = RandomForestRegressor()

# Treina o modelo de Rando Forest
RF.fit(x_treino, y_treino)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


6 - Avaliação Estatística dos Resultados.

In [27]:
# Calcula as Métricas estatísticas.
RF_mbe = mean_absolute_error(y_teste, RF.predict(x_teste))

RF_rmse = mean_squared_error(y_teste, RF.predict(x_teste))

RF_r2 = r2_score(y_teste, RF.predict(x_teste))

# Exibe os Resultados.
print('Resultados do modelo de Random Forest:')
# Erro Médio (MBE).
print(f'MBE: {RF_mbe:.2f}')
# Raiz do erro Quadrático medio (RMSE).
print(f'MSE: {RF_rmse:.2f}')
# Coeficiente de Determinação (R2).
print(f'R²: {RF_r2:.2f}')



Resultados do modelo de Random Forest:
MBE: 3.29
MSE: 20.42
R²: 0.49


7 - Salva o modelo

In [28]:
geo_dict = df_geolocation.set_index(
    "geolocation_zip_code_prefix"
)[["geolocation_lat", "geolocation_lng"]].to_dict("index")

joblib.dump(geo_dict, "../model/geo_dict.pkl")
joblib.dump(RF, "../model/model.pkl")
joblib.dump(encoders, '../model/encoders.pkl')

['../model/encoders.pkl']